# 03 — Explainability

SHAP analysis per spec.md §9.3 + acceptance criterion 6. Reads saved
Model B from `models/`. All plots write to `results/shap/`.

Run AFTER `make train` + `make evaluate` so the pickles and predictions
parquets are fresh. Notebook is feature-set-agnostic — works whether
Model B has the 10-column original SA2 block or the 29-column broadened
block.

## Setup

Load features, models, and feature lists. Sample 10k rows from the
`test_normal` fold for SHAP — `TreeExplainer` is fast but 850k rows is
overkill (mean |SHAP| stabilises well before then). Seeded for reproducibility.

In [ ]:
from __future__ import annotations

import json
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

from fuel_pred import config

# ---- features + test_normal fold ----------------------------------------
features = pd.read_parquet(config.DATA_PROCESSED / "features.parquet")
features["date"] = pd.to_datetime(features["date"])
mask_test = (
    (features["date"] >= config.TEST_START)
    & (features["date"] <= config.TEST_NORMAL_END)
)
test = features.loc[mask_test].copy()
print(f"test_normal: {len(test):,} rows, {test['station_id'].nunique():,} stations")

# ---- models + feature lists ---------------------------------------------
with open(config.MODELS_DIR / "model_a.pkl", "rb") as fh:
    model_a = pickle.load(fh)
with open(config.MODELS_DIR / "model_b.pkl", "rb") as fh:
    model_b = pickle.load(fh)
with open(config.MODELS_DIR / "feature_lists.json") as fh:
    feature_lists = json.load(fh)

cols_a: list[str] = feature_lists["A"]["feature_columns"]
cols_b: list[str] = feature_lists["B"]["feature_columns"]
n_sa2 = sum(1 for c in cols_b if c.startswith("sa2_"))
print(f"Model A: {len(cols_a)} features")
print(f"Model B: {len(cols_b)} features ({n_sa2} sa2_*)")

# ---- SHAP sample --------------------------------------------------------
SAMPLE_N = 10_000
rng = np.random.default_rng(seed=42)
sample_idx = rng.choice(len(test), size=min(SAMPLE_N, len(test)), replace=False)
test_sample = test.iloc[sample_idx].reset_index(drop=True)

# ---- restore categorical dtypes from the model --------------------------
# features.parquet round-trips strings as object dtype. LightGBM was
# trained with the §7 categorical columns as pd.Categorical and stored
# the per-column category sets in `booster_.pandas_categorical` (in
# feature-column order, categorical cols only). Without re-applying
# those categories here, model.predict() / SHAP raises:
#   ValueError: train and valid dataset categorical_feature do not match.
from fuel_pred.train.feature_blocks import CATEGORICAL_COLUMNS

categorical_cols_b = [c for c in cols_b if c in CATEGORICAL_COLUMNS]
if len(categorical_cols_b) != len(model_b.booster_.pandas_categorical):
    raise RuntimeError(
        f"Categorical count mismatch: model has "
        f"{len(model_b.booster_.pandas_categorical)} pandas_categorical entries "
        f"but Model B feature list has {len(categorical_cols_b)} categorical columns. "
        f"CATEGORICAL_COLUMNS in feature_blocks.py may have drifted from training."
    )
for col, cats in zip(categorical_cols_b, model_b.booster_.pandas_categorical):
    test_sample[col] = pd.Categorical(test_sample[col], categories=cats)
print(f"restored {len(categorical_cols_b)} categorical column(s): {categorical_cols_b}")

X_b_sample = test_sample[cols_b]
print(f"SHAP sample: {len(test_sample):,} rows")

# ---- output dir ---------------------------------------------------------
config.SHAP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Outputs → {config.SHAP_DIR}")

In [ ]:
# Compute SHAP values once for the sample — reused across sections 1-3 and 5.
# TreeExplainer on LightGBM is the recommended path: exact (not approximated)
# Shapley values, vectorised, ~30-60s for 10k x ~85 features.
explainer_b = shap.TreeExplainer(model_b)
shap_values_b = explainer_b.shap_values(X_b_sample)
print(f"SHAP values shape: {shap_values_b.shape}")
print(f"expected_value (base): {explainer_b.expected_value:.3f}")

## 1. SHAP summary plot — Model B (top 30 features)

Each point is one row × one feature; x-axis is SHAP value (impact on
prediction in cents/L); colour is the feature's value (red = high,
blue = low). Features ranked by mean |SHAP| over the sample.

Saves to `results/shap/summary_b.png`.

In [ ]:
plt.figure(figsize=(10, 12))
shap.summary_plot(
    shap_values_b,
    X_b_sample,
    plot_type="dot",
    max_display=30,
    show=False,
)
plt.title(
    f"Model B \u2014 SHAP feature impact (top 30 of {len(cols_b)}) "
    f"on a {len(test_sample):,}-row test_normal sample"
)
plt.tight_layout()
plt.savefig(config.SHAP_DIR / "summary_b.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. SHAP dependence plots for top SA2 features

One plot per top-5 `sa2_*` feature, ranked by mean |SHAP| within that
block. Each plot shows how the feature's value maps to its SHAP impact,
with auto-detected interaction partner shown in colour.

Saves one PNG per feature to `results/shap/dependence_<feature>.png`.

In [ ]:
sa2_indices = [i for i, c in enumerate(cols_b) if c.startswith("sa2_")]
if not sa2_indices:
    raise RuntimeError(
        "Model B has no sa2_* features \u2014 either feature_blocks.SA2_COLUMNS "
        "or features.parquet doesn't include the augmentor block."
    )

sa2_mean_abs = np.abs(shap_values_b[:, sa2_indices]).mean(axis=0)
ranking = pd.DataFrame(
    {
        "feature": [cols_b[i] for i in sa2_indices],
        "mean_abs_shap": sa2_mean_abs,
    }
).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
print("All sa2_* features ranked by mean |SHAP|:")
print(ranking.to_string(index=False))

TOP_N = 5
for feat in ranking["feature"].head(TOP_N):
    plt.figure(figsize=(8, 5))
    shap.dependence_plot(
        feat,
        shap_values_b,
        X_b_sample,
        interaction_index="auto",
        show=False,
    )
    plt.title(f"SHAP dependence: {feat}")
    plt.tight_layout()
    safe = feat.replace("/", "_")
    plt.savefig(
        config.SHAP_DIR / f"dependence_{safe}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()

## 3. SHAP interaction plot \u2b50

`cal_day_of_fortnight \xd7 sa2_seifa_irsd_score` \u2014 the demonstration of
the augmentor's interaction value. If the SA2 block is teaching the
model that disadvantaged-area stations have a different fortnight-cycle
shape than advantaged-area stations, this plot is where it shows up.

Saves to `results/shap/interaction_dof_seifa.png`.

In [ ]:
KEY = "cal_day_of_fortnight"
PARTNER = "sa2_seifa_irsd_score"

missing = [c for c in (KEY, PARTNER) if c not in cols_b]
if missing:
    raise RuntimeError(
        f"Cannot draw interaction plot \u2014 missing from Model B feature set: {missing}"
    )

plt.figure(figsize=(10, 6))
shap.dependence_plot(
    KEY,
    shap_values_b,
    X_b_sample,
    interaction_index=PARTNER,
    show=False,
)
plt.title(
    f"SHAP interaction: {KEY} \xd7 {PARTNER}\n"
    f"(Centrelink-fortnight pattern modulated by socio-economic disadvantage)"
)
plt.tight_layout()
plt.savefig(
    config.SHAP_DIR / "interaction_dof_seifa.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

## 4. Comparison of top-20 feature importances: Model A vs Model B

Side-by-side bars. SA2 features highlighted in orange so it's visually
obvious where (and whether) the augmentor block displaces non-SA2
features from Model B's top-20.

Uses gain-importance (LightGBM's own), matching the table in
`results/comparison.md`. Saves to `results/shap/importance_a_vs_b.png`.

In [ ]:
def _gain_importance(model, cols: list[str]) -> pd.Series:
    g = model.booster_.feature_importance(importance_type="gain")
    return pd.Series(g, index=cols).sort_values(ascending=False)

imp_a = _gain_importance(model_a, cols_a)
imp_b = _gain_importance(model_b, cols_b)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))
for ax, imp, label in zip(
    axes,
    [imp_a, imp_b],
    [f"Model A \u2014 no SA2 ({len(cols_a)} features)",
     f"Model B \u2014 with SA2 ({len(cols_b)} features)"],
):
    top20 = imp.head(20).iloc[::-1]  # reverse so top is at the top of the bar chart
    colors = [
        "tab:orange" if c.startswith("sa2_") else "tab:blue"
        for c in top20.index
    ]
    ax.barh(top20.index, top20.values, color=colors)
    ax.set_title(f"{label}\nTop 20 by gain importance")
    ax.set_xlabel("gain")
    ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(config.SHAP_DIR / "importance_a_vs_b.png", dpi=150, bbox_inches="tight")
plt.show()

sa2_top20 = [c for c in imp_b.head(20).index if c.startswith("sa2_")]
print(f"\nSA2 features in Model B top 20 ({len(sa2_top20)}): {sa2_top20}")

## 5. Per-station case studies

Three stations spanning the SEIFA gradient (Q1 / Q3 / Q5 quintiles).
For each:

1. Predictions vs actuals over the `test_normal` fold (both models).
2. SHAP waterfall for one representative day, showing which features
   pushed the prediction up vs down.

Saves `results/shap/case_studies_predictions.png` and one
`results/shap/waterfall_<quintile>_<station>.png` per station.

In [ ]:
# ---- Pick one representative station per Q1 / Q3 / Q5 SEIFA quintile ----
station_seifa = (
    features.dropna(subset=["sa2_seifa_irsd_score"])
    .groupby("station_id", as_index=False)["sa2_seifa_irsd_score"].first()
)
station_seifa["q"] = pd.qcut(
    station_seifa["sa2_seifa_irsd_score"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
)
pick_ids: dict[str, str] = {}
for q in ("Q1", "Q3", "Q5"):
    sub = station_seifa[station_seifa["q"] == q].sort_values("sa2_seifa_irsd_score")
    pick_ids[q] = sub.iloc[len(sub) // 2]["station_id"]  # median-of-quintile
print("Picked stations:")
for q, sid in pick_ids.items():
    score = station_seifa.set_index("station_id").loc[sid, "sa2_seifa_irsd_score"]
    print(f"  {q}: {sid}  (SEIFA IRSD score = {score:.0f})")

# ---- Predictions vs actuals plot ---------------------------------------
preds = pd.read_parquet(config.MODELS_DIR / "predictions_test_normal.parquet")
preds["date"] = pd.to_datetime(preds["date"])

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)
for ax, (q, sid) in zip(axes, pick_ids.items()):
    sub = preds[preds["station_id"] == sid].sort_values("date")
    ax.plot(sub["date"], sub["y_true"], "-", lw=1.2, color="k", label="actual")
    ax.plot(sub["date"], sub["y_pred_a"], "-", lw=0.9, alpha=0.75, label="Model A")
    ax.plot(sub["date"], sub["y_pred_b"], "-", lw=0.9, alpha=0.75, label="Model B")
    mae_a = (sub["y_pred_a"] - sub["y_true"]).abs().mean()
    mae_b = (sub["y_pred_b"] - sub["y_true"]).abs().mean()
    ax.set_title(
        f"{q} \u2014 station {sid}  "
        f"(MAE: A={mae_a:.2f}, B={mae_b:.2f}, \u0394={mae_b - mae_a:+.2f})"
    )
    ax.set_ylabel("U91 price (cents/L)")
    ax.legend(loc="upper left", fontsize=8)
axes[-1].set_xlabel("date")
plt.tight_layout()
plt.savefig(
    config.SHAP_DIR / "case_studies_predictions.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

# ---- SHAP waterfall per station ----------------------------------------
# For each station, find a row to explain. Prefer one from our SHAP sample
# (so values are already computed); else compute on the fly.
for q, sid in pick_ids.items():
    in_sample = test_sample.index[test_sample["station_id"] == sid].tolist()
    if in_sample:
        i = in_sample[len(in_sample) // 2]
        row = test_sample.iloc[i]
        shap_row = shap_values_b[i]
    else:
        # Station not in the 10k sample \u2014 compute SHAP just for one row.
        row_full = test[test["station_id"] == sid].iloc[len(test[test["station_id"] == sid]) // 2]
        shap_row = explainer_b.shap_values(row_full[cols_b].to_frame().T)[0]
        row = row_full

    expl = shap.Explanation(
        values=shap_row,
        base_values=explainer_b.expected_value,
        data=row[cols_b].to_numpy(),
        feature_names=cols_b,
    )
    plt.figure(figsize=(10, 9))
    shap.plots.waterfall(expl, max_display=15, show=False)
    plt.title(
        f"SHAP waterfall \u2014 {q} station {sid} on {row['date'].date()}\n"
        f"y_true={row.get('y_t1', float('nan')):.2f}, "
        f"\u0394 from base = {shap_row.sum():+.2f}"
    )
    plt.tight_layout()
    plt.savefig(
        config.SHAP_DIR / f"waterfall_{q}_{sid}.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()